# Feature Extraction Notebook: TF-IDF and Word2Vec

## Task Description

This notebook implements two feature extraction methods on a processed text dataset (`processed_text.csv`):

1. **TF-IDF** using `TfidfVectorizer` from `scikit-learn`
2. **Word2Vec** using `gensim`

### Objectives

- Load and validate required columns: `processed_text`, `tokens`, and `sentiment`
- Handle missing values safely
- Generate TF-IDF vectors (`max_features=5000`) and persist outputs
- Train a Word2Vec model (`vector_size=100`, `window=5`, `min_count=2`, `workers=4`)
- Build sentence-level embeddings via average word vectors
- Export all feature artifacts into `.pkl` files for downstream ML tasks

## 1) Setup and Libraries

In [2]:
# Install gensim
!pip install gensim

# Import libraries
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from gensim.models import Word2Vec

print("Libraries imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.1 MB/s eta 0:00:00
Libraries imported successfully.


## 2) Data Loading & Validation

In [3]:
# Path to processed dataset
DATA_PATH = "processed_text.csv"

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded successfully. Shape: {df.shape}")

# Validate required columns
required_columns = {"processed_text", "tokens", "sentiment"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Handle missing values for key text fields
df["processed_text"] = df["processed_text"].fillna("").astype(str)
df["tokens"] = df["tokens"].fillna("").astype(str)

# Drop rows with empty processed text after cleanup
df = df[df["processed_text"].str.strip() != ""].copy()
df.reset_index(drop=True, inplace=True)

print("Validation complete.")
print(df[["processed_text", "tokens", "sentiment"]].head())

Dataset loaded successfully. Shape: (49582, 3)
Validation complete.
                                      processed_text  \
0  reviewer mention watch episode youll hook righ...   
1  wonderful little production film technique una...   
2  think wonderful way spend time hot summer week...   
3  basically family little boy jake think zombie ...   
4  petter matteis love time money visually stun f...   

                                              tokens sentiment  
0  reviewer mention watch episode youll hook righ...  positive  
1  wonderful little production film technique una...  positive  
2  think wonderful way spend time hot summer week...  positive  
3  basically family little boy jake think zombie ...  negative  
4  petter matteis love time money visually stun f...  positive  


## 3) TF-IDF Feature Extraction

The TF-IDF method converts cleaned text into numeric vectors that capture term importance within each document relative to the full corpus.

In [4]:
# Initialize TF-IDF vectorizer with the required maximum vocabulary size
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit and transform the processed text to create design matrix X
tfidf_matrix = tfidf_vectorizer.fit_transform(df["processed_text"])

# Target labels y
y = df["sentiment"].to_numpy()

# Store TF-IDF metadata dictionary using the X & y rule
tfidf_output = {
    "vectorizer": tfidf_vectorizer,
    "X": tfidf_matrix,
    "feature_names": tfidf_vectorizer.get_feature_names_out(),
    "y": y,
}

with open("tfidf_vectors.pkl", "wb") as f:
    pickle.dump(tfidf_output, f)

# Export vocabulary as plain text (one token per line)
with open("tfidf_vocabulary.txt", "w", encoding="utf-8") as f:
    for term in tfidf_output["feature_names"]:
        f.write(f"{term}\n")

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf_output['feature_names'])}")
print("Saved TF-IDF artifacts to tfidf_vectors.pkl")
print("Saved TF-IDF vocabulary to tfidf_vocabulary.txt")

TF-IDF matrix shape: (49582, 5000)
Vocabulary size: 5000
Saved TF-IDF artifacts to tfidf_vectors.pkl
Saved TF-IDF vocabulary to tfidf_vocabulary.txt


## 4) Word2Vec Word Embeddings

This section trains a Word2Vec model from tokenized text and then computes sentence-level embeddings by averaging token vectors for each review.

In [5]:
# Convert each token string (e.g., "word1 word2") to a list of words
df["token_list"] = df["tokens"].apply(lambda x: x.split())

# Keep non-empty token lists for robust training
token_sequences = [tokens for tokens in df["token_list"] if len(tokens) > 0]

if not token_sequences:
    raise ValueError("No valid token sequences found for Word2Vec training.")

# Train Word2Vec with the requested hyperparameters
w2v_model = Word2Vec(
    sentences=token_sequences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
)

print("Word2Vec model training complete.")
print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")


def average_sentence_vector(tokens, model, vector_size=100):
    """Return the average embedding vector for one tokenized sentence.

    Parameters
    ----------
    tokens : list[str]
        Tokens for a single sentence/review.
    model : gensim.models.Word2Vec
        Trained Word2Vec model.
    vector_size : int, default=100
        Dimension of embedding vectors.

    Returns
    -------
    np.ndarray
        Averaged sentence embedding of shape (vector_size,).
    """
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(vector_size, dtype=float)
    return np.mean(valid_vectors, axis=0)


# Build sentence-level embeddings for all reviews
sentence_embeddings = np.vstack(
    df["token_list"].apply(lambda tokens: average_sentence_vector(tokens, w2v_model, vector_size=100)).to_numpy()
)

# L2-normalize embeddings for stable downstream neural training
sentence_embeddings = normalize(sentence_embeddings, norm="l2")

print(f"Sentence embeddings shape (normalized): {sentence_embeddings.shape}")

Word2Vec model training complete.
Word2Vec vocabulary size: 64530
Sentence embeddings shape (normalized): (49582, 100)


## 5) Exporting Results

Store trained artifacts for reproducibility and easy reuse in model training/inference.

In [6]:
# Persist Word2Vec metadata dictionary using the X & y rule
word2vec_output = {
    "model": w2v_model,
    "X": sentence_embeddings,
    "y": y,
}

with open("word2vec_vectors.pkl", "wb") as f:
    pickle.dump(word2vec_output, f)

print("Saved Word2Vec artifacts to word2vec_vectors.pkl")


def print_model_documentation(df, tfidf_data, w2v_data):
    """Print concise model documentation statistics for feature extraction outputs."""
    total_docs = len(df)
    tfidf_shape = tfidf_data["X"].shape
    tfidf_vocab_size = len(tfidf_data["feature_names"])
    w2v_vocab_size = len(w2v_data["model"].wv)
    w2v_shape = w2v_data["X"].shape

    print("\n=== Model Documentation ===")
    print(f"Total documents processed: {total_docs}")
    print(f"TF-IDF matrix shape: {tfidf_shape}")
    print(f"TF-IDF vocabulary size: {tfidf_vocab_size}")
    print(f"Word2Vec vocabulary size: {w2v_vocab_size}")
    print(f"Word2Vec sentence embedding shape: {w2v_shape}")


# Create quick debugging sample file
sample_df = df[["processed_text", "sentiment"]].head(5).copy()
sample_df["embedding_preview"] = [
    np.array2string(vec[:10], precision=4, separator=", ") for vec in sentence_embeddings[:5]
]
sample_df.to_csv("sample_features.csv", index=False)

print("Saved sample debugging file to sample_features.csv")

# Print final report
print_model_documentation(df, tfidf_output, word2vec_output)

# Quick artifact validation
for artifact_path in ["tfidf_vectors.pkl", "word2vec_vectors.pkl"]:
    try:
        with open(artifact_path, "rb") as f:
            _ = pickle.load(f)
        print(f"Verified: {artifact_path}")
    except Exception as e:
        print(f"Validation failed for {artifact_path}: {e}")

Saved Word2Vec artifacts to word2vec_vectors.pkl
Saved sample debugging file to sample_features.csv

=== Model Documentation ===
Total documents processed: 49582
TF-IDF matrix shape: (49582, 5000)
TF-IDF vocabulary size: 5000
Word2Vec vocabulary size: 64530
Word2Vec sentence embedding shape: (49582, 100)
Verified: tfidf_vectors.pkl
Verified: word2vec_vectors.pkl
